# E0 evaluation helper

Mounts the finished E0 kernel output and writes predictions on val_cal and the 5,301 independent
negatives. No training. E0 val_report metrics are already in the E0 kernel output.

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi","-L"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available()

In [ ]:
from pathlib import Path
DATA = sorted(Path("/kaggle/input").rglob("manifest_summary.json"))[0].parent
CODE = sorted(Path("/kaggle/input").rglob("stage_b_train.py"))[0].parent
EVALNEG = sorted(Path("/kaggle/input").rglob("eval_neg.txt"))[0].parent.parent
E0 = sorted(Path("/kaggle/input").rglob("best_recall.pt"))[0]
print("DATA",DATA); print("CODE",CODE); print("EVALNEG",EVALNEG); print("E0",E0)

In [ ]:
from pathlib import Path
def resolve(line, root):
    p=Path(line.strip()); return p if p.is_absolute() else (Path(root)/p).resolve()
out=Path("/kaggle/working/lists"); out.mkdir(parents=True,exist_ok=True)
cal=[resolve(l,DATA) for l in open(DATA/"lists/val_cal.txt",encoding="utf-8") if l.strip()]
indep=[resolve(l,EVALNEG) for l in open(EVALNEG/"lists/eval_neg.txt",encoding="utf-8") if l.strip()]
(out/"val_cal.txt").write_text("\n".join(str(p) for p in cal)+"\n",encoding="utf-8")
(out/"eval_neg.txt").write_text("\n".join(str(p) for p in indep)+"\n",encoding="utf-8")
print(len(cal),len(indep))

In [ ]:
import subprocess
for name, lst in (("cal","/kaggle/working/lists/val_cal.txt"),("indep","/kaggle/working/lists/eval_neg.txt")):
    subprocess.run(["python",str(Path(CODE)/"stage_b_predict.py"),"--model",str(E0),"--list",lst,
                    "--out",f"/kaggle/working/preds_E0_{name}.jsonl","--device","0","--half","--batch","1"],check=True)

In [ ]:
print("E0 predictions written. Download with: python -m kaggle kernels output nidhishreesp/firesight-stage-b-e0-eval -p build/kaggle_e0_eval_out")